# Test03 oncology genes: pretag candidate blocks

Load section-level literature blocks from `00_grab_literature.ipynb`, run entity pretagging, and save normalized/raw drug-gene candidate summaries. This notebook intentionally stops before LLM classification.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm.notebook import tqdm

from dgilit import (
    BioBertEntityTagger,
    CompositeEntityTagger,
    EntityPreTaggingService,
    PubTator3ChemicalTagger,
    SQLiteNormalizerCache,
    TaggerConfig,
    ViccNormalizer,
)

load_dotenv()

PUBTATOR_PATH = os.environ["PUBTATOR_PATH"]

DATA_DIR = Path("../../../data")
LITERATURE_XLSX = DATA_DIR / "2026-07-02-test03-oncology-genes-literature.xlsx"

CANDIDATE_BLOCKS_CSV = Path("2026-07-02-test03-oncology-genes-candidate-blocks.csv")
CANDIDATE_BLOCKS_PARQUET = Path("2026-07-02-test03-oncology-genes-candidate-blocks.parquet")

LITERATURE_XLSX

In [ ]:
df = pd.read_excel(LITERATURE_XLSX)

df["context"] = df["context"].fillna("").astype(str)
df = df[df["context"].str.strip().ne("")].reset_index(drop=True)

print(f"Loaded {len(df)} section-level literature blocks")
print(f"Unique PMIDs: {df['pmid'].nunique()}")

df.head()

In [ ]:
tagger = CompositeEntityTagger(
    BioBertEntityTagger(
        TaggerConfig(
            batch_size=16,
            include_drugs=True,
            include_genes=True,
            include_diseases=False,
            device=-1,
        )
    ),
    PubTator3ChemicalTagger.from_pubtator_file(PUBTATOR_PATH),
)

pretagger = EntityPreTaggingService(
    tagger=tagger,
    normalizer=ViccNormalizer(
        cache=SQLiteNormalizerCache(".dgilit_normalizer_cache.sqlite")
    ),
)

In [ ]:
contexts = df["context"].tolist()
pmids = df["pmid"].astype(str).tolist()
block_ids = df["block_id"].astype(str).tolist()

tagged_blocks = pretagger.tag_blocks(
    contexts=contexts,
    pmids=pmids,
    block_ids=block_ids,
)

len(tagged_blocks)

In [ ]:
def normalized_entity_name(entity):
    if entity.concept and entity.concept.concept_label and entity.concept.concept_id:
        return entity.concept.concept_label
    return None


def raw_entity_name(entity):
    if entity.text:
        return entity.text
    return None


def get_candidates(block):
    normalized_drugs = sorted({
        name
        for entity in block.entities
        if entity.entity_type == "drug"
        for name in [normalized_entity_name(entity)]
        if name
    })

    normalized_genes = sorted({
        name
        for entity in block.entities
        if entity.entity_type == "gene"
        for name in [normalized_entity_name(entity)]
        if name
    })

    raw_drugs = sorted({
        name
        for entity in block.entities
        if entity.entity_type == "drug"
        for name in [raw_entity_name(entity)]
        if name
    })

    raw_genes = sorted({
        name
        for entity in block.entities
        if entity.entity_type == "gene"
        for name in [raw_entity_name(entity)]
        if name
    })

    return normalized_drugs, normalized_genes, raw_drugs, raw_genes

In [ ]:
candidate_rows = []

for row, block in tqdm(
    zip(df.to_dict(orient="records"), tagged_blocks),
    total=len(tagged_blocks),
    desc="Building candidate block table",
):
    normalized_drugs, normalized_genes, raw_drugs, raw_genes = get_candidates(block)

    candidate_rows.append({
        **row,
        "normalized_drugs": ";".join(normalized_drugs),
        "normalized_genes": ";".join(normalized_genes),
        "raw_drugs": ";".join(raw_drugs),
        "raw_genes": ";".join(raw_genes),
        "normalized_drug_count": len(normalized_drugs),
        "normalized_gene_count": len(normalized_genes),
        "raw_drug_count": len(raw_drugs),
        "raw_gene_count": len(raw_genes),
        "entity_count": len(block.entities),
        "should_run_llm": block.should_run_llm,
        "skip_reason": block.skip_reason,
    })

candidate_blocks_df = pd.DataFrame(candidate_rows)

print(candidate_blocks_df.shape)
candidate_blocks_df.head()

In [ ]:
candidate_blocks_df.to_csv(CANDIDATE_BLOCKS_CSV, index=False)

try:
    candidate_blocks_df.to_parquet(CANDIDATE_BLOCKS_PARQUET, index=False)
except ImportError as exc:
    print(f"Skipping parquet export because an optional parquet engine is unavailable: {exc}")

print(CANDIDATE_BLOCKS_CSV)
print(CANDIDATE_BLOCKS_PARQUET)